# Job Listings Search Engine
### Data Mining Lab Project
**Approach:** Inverted Index + TF-IDF Ranking  
**Language:** Python  
**Domain:** Job Listings  

---
Run each cell **top to bottom** (Runtime > Run all).

In [ ]:
# ============================================================
# CELL 1 - Install and Import Dependencies
# Run this first
# ============================================================
!pip install nltk -q

import re, math, nltk
nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

print("All dependencies loaded.")

## Step 1 - Dataset (100 Job Listings)

In [ ]:
# ============================================================
# CELL 2 - Job Listings Dataset
# 100 job postings across various tech and non-tech domains
# ============================================================

JOB_LISTINGS = [
    {"id": 1, "title": "Python Backend Developer", "company": "TechCorp BD", "location": "Dhaka",
     "description": "We are looking for an experienced Python developer to build and maintain backend APIs using Django and FastAPI. Proficiency in SQL and PostgreSQL required. Experience with Docker and REST APIs is a plus.",
     "tags": ["python", "django", "fastapi", "backend", "sql", "postgresql"]},

    {"id": 2, "title": "Data Scientist", "company": "Analytics Hub", "location": "Dhaka",
     "description": "Seeking a data scientist skilled in machine learning, Python, and data analysis. You will build predictive models using scikit-learn and TensorFlow. Knowledge of statistics and data visualization required.",
     "tags": ["data science", "machine learning", "python", "tensorflow", "statistics"]},

    {"id": 3, "title": "Frontend React Developer", "company": "WebWave", "location": "Chittagong",
     "description": "Join our team as a frontend developer with strong skills in React, JavaScript, and CSS. Experience with Redux, REST APIs, and responsive design is essential. TypeScript knowledge is a bonus.",
     "tags": ["react", "javascript", "css", "frontend", "redux", "typescript"]},

    {"id": 4, "title": "Machine Learning Engineer", "company": "AI Ventures", "location": "Dhaka",
     "description": "We need a machine learning engineer to develop and deploy ML models at scale. Proficiency in Python, PyTorch, and TensorFlow required. Knowledge of MLOps and cloud platforms like AWS is preferred.",
     "tags": ["machine learning", "python", "pytorch", "tensorflow", "mlops", "aws"]},

    {"id": 5, "title": "DevOps Engineer", "company": "CloudBase", "location": "Remote",
     "description": "Looking for a DevOps engineer with expertise in AWS, Docker, Kubernetes, and CI/CD pipelines. Experience with Terraform and Linux system administration required.",
     "tags": ["devops", "aws", "docker", "kubernetes", "ci/cd", "terraform", "linux"]},

    {"id": 6, "title": "Full Stack Developer", "company": "StartupX", "location": "Dhaka",
     "description": "Full stack developer needed for a fast-growing startup. Skills required: Node.js, React, MongoDB, and REST API design. Experience with agile development and Git version control preferred.",
     "tags": ["nodejs", "react", "mongodb", "fullstack", "rest", "git"]},

    {"id": 7, "title": "Software QA Engineer", "company": "QualityFirst", "location": "Sylhet",
     "description": "QA engineer required with experience in manual and automated testing. Proficiency in Selenium, pytest, and JIRA needed. Understanding of software development lifecycle and bug tracking required.",
     "tags": ["qa", "testing", "selenium", "pytest", "automation", "jira"]},

    {"id": 8, "title": "Database Administrator", "company": "DataSafe Ltd", "location": "Dhaka",
     "description": "Experienced DBA needed to manage MySQL, PostgreSQL, and MongoDB databases. Responsibilities include performance tuning, backup management, and database security. SQL optimization skills essential.",
     "tags": ["dba", "mysql", "postgresql", "mongodb", "sql", "database"]},

    {"id": 9, "title": "Android Mobile Developer", "company": "AppFactory", "location": "Dhaka",
     "description": "Android developer required with strong Kotlin and Java skills. Experience building apps with Jetpack Compose, Retrofit, and Firebase. Understanding of Material Design and Google Play Store deployment.",
     "tags": ["android", "kotlin", "java", "mobile", "firebase", "jetpack"]},

    {"id": 10, "title": "iOS Developer", "company": "MobileMinds", "location": "Remote",
     "description": "Experienced iOS developer needed with Swift and Objective-C expertise. Must have knowledge of UIKit, SwiftUI, Core Data, and App Store submission process. REST API integration experience required.",
     "tags": ["ios", "swift", "objective-c", "swiftui", "xcode", "mobile"]},

    {"id": 11, "title": "Cybersecurity Analyst", "company": "SecureNet", "location": "Dhaka",
     "description": "Cybersecurity analyst required to monitor and protect network infrastructure. Skills needed: penetration testing, vulnerability assessment, firewall management, and knowledge of OWASP guidelines.",
     "tags": ["cybersecurity", "penetration testing", "network", "firewall", "owasp"]},

    {"id": 12, "title": "UI/UX Designer", "company": "DesignStudio", "location": "Dhaka",
     "description": "Creative UI/UX designer with expertise in Figma, Adobe XD, and user research. Strong portfolio required demonstrating web and mobile design. Knowledge of design systems and accessibility standards.",
     "tags": ["ui", "ux", "design", "figma", "adobe xd", "user research"]},

    {"id": 13, "title": "Data Analyst", "company": "InsightCo", "location": "Chittagong",
     "description": "Data analyst needed with skills in Excel, SQL, Python, and Power BI. Responsibilities include data cleaning, dashboard creation, and reporting. Strong analytical and communication skills required.",
     "tags": ["data analysis", "sql", "python", "power bi", "excel", "reporting"]},

    {"id": 14, "title": "Cloud Architect", "company": "SkyTech", "location": "Remote",
     "description": "Senior cloud architect needed to design and implement AWS and Azure solutions. Experience with cloud migration, microservices, serverless architecture, and cost optimization required.",
     "tags": ["cloud", "aws", "azure", "architect", "microservices", "serverless"]},

    {"id": 15, "title": "NLP Research Engineer", "company": "LangAI", "location": "Dhaka",
     "description": "NLP engineer needed to work on natural language processing models. Experience with transformers, BERT, GPT models, and HuggingFace required. Python and PyTorch skills essential.",
     "tags": ["nlp", "natural language processing", "python", "bert", "transformers", "pytorch"]},

    {"id": 16, "title": "Blockchain Developer", "company": "ChainTech", "location": "Remote",
     "description": "Blockchain developer needed with experience in Solidity, Ethereum, and smart contracts. Knowledge of Web3.js, DeFi protocols, and NFT development. Python or JavaScript background preferred.",
     "tags": ["blockchain", "solidity", "ethereum", "smart contracts", "web3", "nft"]},

    {"id": 17, "title": "Network Engineer", "company": "NetSolutions", "location": "Dhaka",
     "description": "Network engineer required to design and manage enterprise networks. Expertise in Cisco routers, switches, VPN configuration, and network monitoring tools. CCNA or CCNP certification preferred.",
     "tags": ["networking", "cisco", "vpn", "ccna", "linux", "infrastructure"]},

    {"id": 18, "title": "Product Manager", "company": "ProductHouse", "location": "Dhaka",
     "description": "Experienced product manager needed to lead software product development. Skills required: roadmap planning, stakeholder management, agile, user story writing, and data-driven decision making.",
     "tags": ["product management", "agile", "scrum", "roadmap", "stakeholder"]},

    {"id": 19, "title": "Java Spring Boot Developer", "company": "EnterpriseApps", "location": "Dhaka",
     "description": "Java developer required with Spring Boot, Hibernate, and Microservices experience. Knowledge of REST APIs, Maven, Jenkins CI/CD, and MySQL database required. Experience with Docker is a plus.",
     "tags": ["java", "spring boot", "hibernate", "microservices", "mysql", "docker"]},

    {"id": 20, "title": "Game Developer", "company": "GameZone", "location": "Dhaka",
     "description": "Game developer needed with Unity and C# skills. Experience in 2D and 3D game development, physics simulation, and game optimization. Knowledge of Blender for 3D modeling is a bonus.",
     "tags": ["game development", "unity", "c#", "3d", "blender", "simulation"]},

    {"id": 21, "title": "Business Intelligence Developer", "company": "BIWorks", "location": "Dhaka",
     "description": "BI developer needed with Power BI, Tableau, and SQL expertise. Responsibilities include building dashboards, ETL pipelines, and data warehouse management. Strong analytical skills required.",
     "tags": ["bi", "power bi", "tableau", "sql", "etl", "data warehouse"]},

    {"id": 22, "title": "Embedded Systems Engineer", "company": "HardwarePlus", "location": "Chittagong",
     "description": "Embedded systems engineer required with C and C++ programming experience. Knowledge of microcontrollers, RTOS, IoT, and hardware debugging tools. Experience with Arduino or Raspberry Pi preferred.",
     "tags": ["embedded", "c", "c++", "iot", "microcontroller", "rtos", "arduino"]},

    {"id": 23, "title": "Site Reliability Engineer", "company": "Uptime Inc", "location": "Remote",
     "description": "SRE needed to ensure system reliability, scalability, and performance. Skills: Linux, Python, Kubernetes, monitoring with Prometheus and Grafana, and incident management.",
     "tags": ["sre", "linux", "python", "kubernetes", "prometheus", "grafana"]},

    {"id": 24, "title": "Technical Content Writer", "company": "TechWrite", "location": "Remote",
     "description": "Technical writer needed to create documentation, tutorials, and blog posts for developer audiences. Strong writing skills and understanding of software development, APIs, and cloud technologies required.",
     "tags": ["writing", "documentation", "technical writing", "api", "content"]},

    {"id": 25, "title": "Computer Vision Engineer", "company": "VisionAI", "location": "Dhaka",
     "description": "Computer vision engineer required with expertise in OpenCV, deep learning, and image processing. Experience with YOLO, CNN models, and TensorFlow or PyTorch required. Python skills essential.",
     "tags": ["computer vision", "opencv", "deep learning", "yolo", "cnn", "python"]},

    {"id": 26, "title": "Scrum Master", "company": "AgilePro", "location": "Dhaka",
     "description": "Certified Scrum Master needed to facilitate agile ceremonies and coach development teams. Experience with JIRA, sprint planning, retrospectives, and stakeholder communication required.",
     "tags": ["scrum", "agile", "jira", "sprint", "project management"]},

    {"id": 27, "title": "Ruby on Rails Developer", "company": "RailsHub", "location": "Remote",
     "description": "Ruby on Rails developer needed with strong Ruby skills. Experience with MVC architecture, PostgreSQL, REST API development, and RSpec testing. Knowledge of Redis and Sidekiq preferred.",
     "tags": ["ruby", "rails", "postgresql", "rest", "rspec", "redis"]},

    {"id": 28, "title": "Systems Analyst", "company": "AnalyzeIT", "location": "Dhaka",
     "description": "Systems analyst required to analyze business requirements and design IT solutions. Skills needed: UML diagrams, SQL, requirement gathering, and process optimization. Communication skills essential.",
     "tags": ["systems analysis", "uml", "sql", "requirements", "business analysis"]},

    {"id": 29, "title": "AR/VR Developer", "company": "ImmerseTech", "location": "Dhaka",
     "description": "AR/VR developer needed with Unity, Unreal Engine, and C# or C++ skills. Experience developing immersive experiences for HoloLens, Oculus, or mobile AR using ARKit or ARCore.",
     "tags": ["ar", "vr", "unity", "unreal engine", "c#", "arkit", "arcore"]},

    {"id": 30, "title": "IT Support Specialist", "company": "HelpDesk Pro", "location": "Dhaka",
     "description": "IT support specialist needed for first and second-level technical support. Skills: Windows and Linux troubleshooting, networking basics, Active Directory, and ticketing systems like ServiceNow.",
     "tags": ["it support", "windows", "linux", "networking", "active directory", "helpdesk"]},

    {"id": 31, "title": "Data Engineer", "company": "PipelineWorks", "location": "Dhaka",
     "description": "Data engineer needed to build and maintain scalable data pipelines. Experience with Apache Spark, Kafka, Airflow, and cloud data warehouses like BigQuery or Redshift. Python and SQL skills required.",
     "tags": ["data engineering", "spark", "kafka", "airflow", "python", "sql", "bigquery"]},

    {"id": 32, "title": "Golang Backend Developer", "company": "GoTech", "location": "Remote",
     "description": "Backend developer with Go (Golang) experience needed. Must know concurrency patterns, RESTful API design, gRPC, and PostgreSQL. Experience with microservices and Docker containers is essential.",
     "tags": ["golang", "go", "backend", "grpc", "microservices", "docker", "postgresql"]},

    {"id": 33, "title": "Salesforce Developer", "company": "CRMPro", "location": "Dhaka",
     "description": "Salesforce developer with Apex, Visualforce, and Lightning Web Components experience. Knowledge of Salesforce CRM, SOQL, and integration with third-party APIs required.",
     "tags": ["salesforce", "apex", "lwc", "crm", "soql", "lightning"]},

    {"id": 34, "title": "WordPress Developer", "company": "WebPress BD", "location": "Dhaka",
     "description": "WordPress developer required to build and customize websites using themes, plugins, and custom PHP code. Knowledge of WooCommerce, SEO basics, and website performance optimization preferred.",
     "tags": ["wordpress", "php", "woocommerce", "css", "seo", "web"]},

    {"id": 35, "title": "Flutter Mobile Developer", "company": "CrossPlatform", "location": "Dhaka",
     "description": "Flutter developer needed to build cross-platform mobile apps for iOS and Android. Strong Dart programming skills required. Experience with Firebase, REST APIs, and state management like BLoC or Provider.",
     "tags": ["flutter", "dart", "mobile", "firebase", "ios", "android", "bloc"]},

    {"id": 36, "title": "Digital Marketing Specialist", "company": "GrowthHQ", "location": "Dhaka",
     "description": "Digital marketing specialist needed with experience in SEO, Google Ads, Facebook Ads, and email marketing. Proficiency in Google Analytics, content marketing strategy, and campaign management required.",
     "tags": ["digital marketing", "seo", "google ads", "facebook ads", "analytics", "content"]},

    {"id": 37, "title": "Data Visualization Analyst", "company": "ChartIQ", "location": "Remote",
     "description": "Data visualization analyst needed to build interactive dashboards using Tableau, Power BI, or D3.js. Must be able to communicate complex data insights clearly. SQL and Python proficiency required.",
     "tags": ["data visualization", "tableau", "power bi", "d3.js", "sql", "python"]},

    {"id": 38, "title": "SAP Consultant", "company": "EnterpriseSoft", "location": "Dhaka",
     "description": "SAP consultant needed with experience in SAP FICO, SAP MM, or SAP SD modules. Ability to configure and customize SAP ERP systems. Business process knowledge and ABAP programming skills preferred.",
     "tags": ["sap", "erp", "fico", "abap", "business process", "enterprise"]},

    {"id": 39, "title": "Robotics Engineer", "company": "RoboTech", "location": "Dhaka",
     "description": "Robotics engineer required with experience in ROS, Python, and C++. Knowledge of robot kinematics, path planning, and sensor integration. Experience with Arduino, Raspberry Pi, or industrial robots preferred.",
     "tags": ["robotics", "ros", "python", "c++", "automation", "arduino", "sensors"]},

    {"id": 40, "title": "Cloud Security Engineer", "company": "SafeCloud", "location": "Remote",
     "description": "Cloud security engineer needed to implement security best practices on AWS and Azure. Experience with IAM, encryption, security auditing, compliance frameworks, and vulnerability management required.",
     "tags": ["cloud security", "aws", "azure", "iam", "compliance", "encryption"]},

    {"id": 41, "title": "React Native Developer", "company": "NativeMobile", "location": "Remote",
     "description": "React Native developer needed to build cross-platform mobile apps. Strong JavaScript and React skills required. Experience with Redux, Firebase, native device APIs, and app store deployment preferred.",
     "tags": ["react native", "javascript", "mobile", "redux", "firebase", "android", "ios"]},

    {"id": 42, "title": "Ethical Hacker", "company": "PenTestPro", "location": "Remote",
     "description": "Ethical hacker needed to perform penetration testing and security assessments. Experience with Metasploit, Burp Suite, Nmap, and Kali Linux required. CEH or OSCP certification is a plus.",
     "tags": ["ethical hacking", "penetration testing", "metasploit", "kali linux", "oscp", "security"]},

    {"id": 43, "title": "PHP Laravel Developer", "company": "LaravelBD", "location": "Dhaka",
     "description": "PHP Laravel developer needed to build web applications. Strong knowledge of Laravel, MySQL, RESTful APIs, and MVC architecture. Familiarity with Vue.js or React for frontend integration preferred.",
     "tags": ["php", "laravel", "mysql", "rest", "backend", "vue", "react"]},

    {"id": 44, "title": "Tableau Developer", "company": "DashCraft", "location": "Dhaka",
     "description": "Tableau developer needed to design and maintain business intelligence dashboards. Strong SQL skills and experience connecting Tableau to various data sources. Data storytelling and presentation skills required.",
     "tags": ["tableau", "bi", "sql", "data visualization", "dashboard", "reporting"]},

    {"id": 45, "title": "Hardware Engineer", "company": "CircuitLabs", "location": "Chittagong",
     "description": "Hardware engineer needed with PCB design, circuit analysis, and FPGA programming experience. Knowledge of Altium Designer, VHDL, and hardware testing methodologies. Embedded C experience preferred.",
     "tags": ["hardware", "pcb", "fpga", "vhdl", "embedded", "altium", "circuit"]},

    {"id": 46, "title": "Deep Learning Researcher", "company": "NeuroLab", "location": "Dhaka",
     "description": "Deep learning researcher needed to develop novel neural network architectures. Strong background in mathematics, PyTorch or TensorFlow, and experience publishing research papers preferred.",
     "tags": ["deep learning", "neural networks", "pytorch", "tensorflow", "research", "python"]},

    {"id": 47, "title": "Angular Frontend Developer", "company": "NgSolutions", "location": "Dhaka",
     "description": "Angular developer needed to build enterprise web applications. Strong TypeScript and Angular skills required. Experience with RxJS, Angular Material, REST API integration, and unit testing with Jasmine.",
     "tags": ["angular", "typescript", "rxjs", "frontend", "javascript", "jasmine"]},

    {"id": 48, "title": "Technical Project Manager", "company": "BuildRight", "location": "Dhaka",
     "description": "Technical project manager needed to oversee software development projects. PMP or Prince2 certification preferred. Skills: risk management, budgeting, agile, stakeholder communication, and delivery planning.",
     "tags": ["project management", "pmp", "agile", "scrum", "risk management", "leadership"]},

    {"id": 49, "title": "MLOps Engineer", "company": "ModelOps", "location": "Remote",
     "description": "MLOps engineer needed to manage machine learning model deployment and monitoring pipelines. Experience with MLflow, Kubeflow, Docker, Kubernetes, and cloud platforms required.",
     "tags": ["mlops", "mlflow", "kubeflow", "docker", "kubernetes", "machine learning", "python"]},

    {"id": 50, "title": "Vue.js Developer", "company": "VueCraft", "location": "Remote",
     "description": "Vue.js developer required to build modern web applications. Strong JavaScript knowledge and experience with Vuex, Vue Router, REST APIs, and unit testing. Nuxt.js experience is a bonus.",
     "tags": ["vue", "vuex", "javascript", "nuxt", "frontend", "rest"]},

    {"id": 51, "title": "Business Analyst", "company": "BizLogic", "location": "Dhaka",
     "description": "Business analyst needed to bridge the gap between business needs and technical solutions. Skills: requirement gathering, process mapping, stakeholder interviews, use case writing, and SQL for data queries.",
     "tags": ["business analysis", "requirements", "sql", "process mapping", "uml", "communication"]},

    {"id": 52, "title": "ERP Implementation Specialist", "company": "ERPWise", "location": "Dhaka",
     "description": "ERP specialist needed to implement and customize ERP solutions. Experience with Oracle ERP, SAP, or Odoo required. Ability to train users and provide post-implementation support.",
     "tags": ["erp", "oracle", "sap", "odoo", "implementation", "training"]},

    {"id": 53, "title": "Cloud Data Engineer", "company": "CloudStream", "location": "Remote",
     "description": "Cloud data engineer needed to design data lake and warehouse solutions on AWS or GCP. Skills: Python, SQL, dbt, Spark, and orchestration tools like Airflow or Prefect.",
     "tags": ["cloud", "data engineering", "aws", "gcp", "dbt", "spark", "airflow"]},

    {"id": 54, "title": "IT Auditor", "company": "AuditSure", "location": "Dhaka",
     "description": "IT auditor needed to assess information systems risk, compliance, and controls. CISA certification preferred. Knowledge of ISO 27001, COBIT, and IT audit methodologies required.",
     "tags": ["it audit", "cisa", "iso 27001", "compliance", "risk", "governance"]},

    {"id": 55, "title": "Software Architect", "company": "ArchDesign", "location": "Dhaka",
     "description": "Software architect needed to design scalable and maintainable system architectures. Experience with microservices, event-driven architecture, domain-driven design, and cloud platforms required.",
     "tags": ["architecture", "microservices", "event-driven", "ddd", "cloud", "system design"]},

    {"id": 56, "title": "GIS Developer", "company": "MapTech BD", "location": "Dhaka",
     "description": "GIS developer needed to develop geospatial applications. Skills: ArcGIS, QGIS, PostGIS, Python, and web mapping with Leaflet or Mapbox. Experience with spatial data analysis preferred.",
     "tags": ["gis", "arcgis", "qgis", "postgis", "python", "leaflet", "geospatial"]},

    {"id": 57, "title": "Automation Test Engineer", "company": "TestBot", "location": "Remote",
     "description": "Automation test engineer needed to build test frameworks using Selenium, Cypress, or Playwright. Experience with CI/CD integration, API testing with Postman or RestAssured, and BDD frameworks like Cucumber.",
     "tags": ["automation testing", "selenium", "cypress", "playwright", "api testing", "bdd"]},

    {"id": 58, "title": "Hadoop Developer", "company": "BigDataBD", "location": "Dhaka",
     "description": "Hadoop developer needed with experience in MapReduce, HDFS, Hive, and Pig. Knowledge of Spark, HBase, and Kafka for real-time data processing. Java or Python programming skills required.",
     "tags": ["hadoop", "mapreduce", "hive", "spark", "hdfs", "bigdata", "java"]},

    {"id": 59, "title": "Penetration Tester", "company": "RedTeam BD", "location": "Dhaka",
     "description": "Penetration tester needed to identify vulnerabilities in web, mobile, and network systems. Tools: Burp Suite, Nessus, Metasploit, Wireshark. CEH or OSCP certification strongly preferred.",
     "tags": ["penetration testing", "security", "burp suite", "metasploit", "nessus", "oscp"]},

    {"id": 60, "title": "Drupal Developer", "company": "CMSPro", "location": "Remote",
     "description": "Drupal developer needed to build and maintain content management systems. Strong PHP and Drupal module development skills required. Experience with MySQL, RESTful APIs, and theming with Twig preferred.",
     "tags": ["drupal", "php", "cms", "mysql", "twig", "rest", "web"]},

    {"id": 61, "title": "Data Governance Analyst", "company": "DataTrust", "location": "Dhaka",
     "description": "Data governance analyst needed to implement data quality, lineage, and cataloging initiatives. Experience with Collibra, Alation, or Apache Atlas. SQL and data management best practices knowledge required.",
     "tags": ["data governance", "data quality", "collibra", "sql", "metadata", "compliance"]},

    {"id": 62, "title": "Scala Developer", "company": "FuncProg", "location": "Remote",
     "description": "Scala developer needed for big data and distributed systems work. Experience with Apache Spark, Akka, Play Framework, and functional programming concepts required. Java background is acceptable.",
     "tags": ["scala", "spark", "akka", "functional programming", "big data", "java"]},

    {"id": 63, "title": "Low Code Developer", "company": "AppQuick", "location": "Dhaka",
     "description": "Low-code developer needed with experience in OutSystems, Mendix, or Microsoft Power Apps. Ability to build business applications rapidly. Basic SQL and REST API knowledge preferred.",
     "tags": ["low code", "outsystems", "mendix", "power apps", "sql", "business apps"]},

    {"id": 64, "title": "Kubernetes Administrator", "company": "ContainerOps", "location": "Remote",
     "description": "Kubernetes administrator needed to manage and scale container orchestration clusters. Experience with Helm, service mesh like Istio, CI/CD pipelines, and monitoring with Prometheus and Grafana.",
     "tags": ["kubernetes", "helm", "istio", "docker", "devops", "prometheus", "grafana"]},

    {"id": 65, "title": "Quantum Computing Researcher", "company": "QuantumLab", "location": "Remote",
     "description": "Quantum computing researcher needed with background in quantum algorithms, linear algebra, and physics. Experience with Qiskit or Cirq and programming in Python. PhD or Masters in related field preferred.",
     "tags": ["quantum computing", "qiskit", "python", "algorithms", "research", "physics"]},

    {"id": 66, "title": "Marketing Data Analyst", "company": "AdMetrics", "location": "Dhaka",
     "description": "Marketing data analyst needed to measure campaign performance and ROI. Skills: Google Analytics, SQL, Excel, Python for data analysis, A/B testing, and dashboard creation in Tableau or Power BI.",
     "tags": ["marketing analytics", "google analytics", "sql", "python", "a/b testing", "tableau"]},

    {"id": 67, "title": "Rust Systems Developer", "company": "SystemsLab", "location": "Remote",
     "description": "Rust developer needed for systems programming and performance-critical applications. Experience with memory safety, concurrency, WebAssembly, and CLI tool development. C or C++ background is helpful.",
     "tags": ["rust", "systems programming", "webassembly", "concurrency", "performance", "cli"]},

    {"id": 68, "title": "Infrastructure Engineer", "company": "InfraStack", "location": "Remote",
     "description": "Infrastructure engineer needed to manage on-premise and cloud infrastructure. Skills: Terraform, Ansible, Linux server administration, AWS or Azure, and networking. Experience with monitoring tools preferred.",
     "tags": ["infrastructure", "terraform", "ansible", "linux", "aws", "azure", "devops"]},

    {"id": 69, "title": "COBOL Developer", "company": "BankSys", "location": "Dhaka",
     "description": "COBOL developer needed to maintain and modernize legacy banking systems. Experience with COBOL, JCL, DB2, and mainframe environments required. CICS and batch processing knowledge preferred.",
     "tags": ["cobol", "mainframe", "jcl", "db2", "banking", "legacy", "cics"]},

    {"id": 70, "title": "Database Developer", "company": "QueryCraft", "location": "Dhaka",
     "description": "Database developer needed to write complex queries, stored procedures, and database design. Strong skills in SQL Server, T-SQL, query optimization, indexing, and SSRS report creation required.",
     "tags": ["sql server", "t-sql", "database", "ssrs", "stored procedures", "sql"]},

    {"id": 71, "title": "Graph Database Developer", "company": "GraphWorks", "location": "Remote",
     "description": "Graph database developer needed with Neo4j and Cypher query language expertise. Experience with graph algorithms, knowledge graphs, and recommendation systems. Python or Java programming required.",
     "tags": ["neo4j", "graph database", "cypher", "knowledge graph", "python", "recommendation"]},

    {"id": 72, "title": "AI Ethics Researcher", "company": "EthicsAI", "location": "Remote",
     "description": "AI ethics researcher needed to study fairness, bias, transparency, and accountability in machine learning systems. Background in philosophy, computer science, or social science with ML knowledge preferred.",
     "tags": ["ai ethics", "fairness", "bias", "machine learning", "research", "transparency"]},

    {"id": 73, "title": "Cloud FinOps Analyst", "company": "SpendWise", "location": "Remote",
     "description": "Cloud FinOps analyst needed to monitor and optimize cloud spending on AWS, Azure, or GCP. Skills: cost allocation, reserved instances, rightsizing, tagging strategy, and cloud billing analysis.",
     "tags": ["finops", "cloud cost", "aws", "azure", "gcp", "cost optimization"]},

    {"id": 74, "title": "Magento Developer", "company": "EcomBD", "location": "Dhaka",
     "description": "Magento developer needed to build and maintain e-commerce stores. Strong PHP and Magento 2 skills required. Experience with MySQL, REST and GraphQL APIs, and payment gateway integration preferred.",
     "tags": ["magento", "php", "ecommerce", "mysql", "graphql", "payment gateway"]},

    {"id": 75, "title": "Healthcare IT Analyst", "company": "MedTech BD", "location": "Dhaka",
     "description": "Healthcare IT analyst needed to implement and support hospital information systems. Knowledge of HL7, FHIR, EMR/EHR systems, and healthcare data standards. SQL and analytical skills required.",
     "tags": ["healthcare it", "hl7", "fhir", "ehr", "sql", "health informatics"]},

    {"id": 76, "title": "Reinforcement Learning Engineer", "company": "RLab", "location": "Remote",
     "description": "Reinforcement learning engineer needed to develop RL agents for decision making and game playing. Experience with OpenAI Gym, Stable Baselines, PyTorch, and deep Q-networks required.",
     "tags": ["reinforcement learning", "openai gym", "pytorch", "deep learning", "python", "rl"]},

    {"id": 77, "title": "Network Security Engineer", "company": "ShieldNet", "location": "Dhaka",
     "description": "Network security engineer needed to design and implement network defenses. Skills: firewall configuration, IDS/IPS, VPN, zero-trust architecture, and SIEM tools like Splunk or IBM QRadar.",
     "tags": ["network security", "firewall", "siem", "splunk", "zero trust", "vpn", "ids"]},

    {"id": 78, "title": "Python Data Scientist", "company": "NumericLab", "location": "Dhaka",
     "description": "Python data scientist needed to analyze large datasets and build predictive models. Skills: pandas, NumPy, scikit-learn, matplotlib, seaborn, and SQL. Experience with Jupyter notebooks required.",
     "tags": ["python", "pandas", "numpy", "scikit-learn", "data science", "jupyter", "sql"]},

    {"id": 79, "title": "3D Artist", "company": "PixelForge", "location": "Dhaka",
     "description": "3D artist needed to create high-quality 3D models, textures, and animations for games and visualization. Proficiency in Blender, Maya, or 3ds Max. Experience with PBR texturing and Unreal Engine preferred.",
     "tags": ["3d modeling", "blender", "maya", "animation", "texturing", "unreal engine"]},

    {"id": 80, "title": "Financial Software Developer", "company": "FinCode", "location": "Dhaka",
     "description": "Financial software developer needed to build trading and fintech applications. Skills: Python or Java, REST APIs, database design, algorithmic trading concepts, and regulatory compliance knowledge.",
     "tags": ["fintech", "python", "java", "trading", "rest api", "financial software"]},

    {"id": 81, "title": "Supply Chain Analyst", "company": "LogiData", "location": "Dhaka",
     "description": "Supply chain analyst needed to optimize logistics and inventory processes using data. Skills: Excel, SQL, Power BI, demand forecasting, and ERP systems like SAP or Oracle.",
     "tags": ["supply chain", "logistics", "sql", "excel", "power bi", "erp", "forecasting"]},

    {"id": 82, "title": "Bioinformatics Engineer", "company": "GenomicsBD", "location": "Dhaka",
     "description": "Bioinformatics engineer needed to analyze genomic and proteomic datasets. Skills: Python, R, Biopython, sequence alignment, and statistical analysis. Background in biology or computational biology preferred.",
     "tags": ["bioinformatics", "python", "r", "genomics", "biopython", "statistics"]},

    {"id": 83, "title": "CRM Developer", "company": "CustomerTech", "location": "Dhaka",
     "description": "CRM developer needed to customize and integrate CRM platforms like HubSpot, Zoho CRM, or Dynamics 365. REST API integration, SQL, and workflow automation experience required.",
     "tags": ["crm", "hubspot", "zoho", "dynamics 365", "rest api", "sql", "automation"]},

    {"id": 84, "title": "UX Researcher", "company": "InsightDesign", "location": "Dhaka",
     "description": "UX researcher needed to conduct user interviews, usability testing, and surveys. Ability to synthesize findings and present actionable insights to product teams. Experience with tools like Maze or UserTesting preferred.",
     "tags": ["ux research", "usability testing", "user interviews", "design", "product", "research"]},

    {"id": 85, "title": "Telecommunications Engineer", "company": "TelecomBD", "location": "Dhaka",
     "description": "Telecom engineer needed with experience in 4G/5G networks, VoIP, and telecom protocols like SIP and SS7. Network planning, RF optimization, and OSS/BSS systems knowledge required.",
     "tags": ["telecom", "5g", "voip", "sip", "networking", "rf optimization", "oss"]},

    {"id": 86, "title": "Agile Coach", "company": "AgileWave", "location": "Remote",
     "description": "Agile coach needed to guide organizations in agile transformation. Strong knowledge of Scrum, Kanban, SAFe, and lean principles. Coaching and facilitation skills essential. ICP-ACC or similar certification preferred.",
     "tags": ["agile coaching", "scrum", "kanban", "safe", "lean", "transformation", "facilitation"]},

    {"id": 87, "title": "Knowledge Graph Engineer", "company": "SemanticAI", "location": "Remote",
     "description": "Knowledge graph engineer needed to build and maintain semantic knowledge bases. Experience with RDF, SPARQL, OWL ontologies, and graph databases like Neo4j or Amazon Neptune required.",
     "tags": ["knowledge graph", "rdf", "sparql", "ontology", "neo4j", "semantic web"]},

    {"id": 88, "title": "EdTech Developer", "company": "LearnBD", "location": "Dhaka",
     "description": "EdTech developer needed to build online learning platforms. Skills: Python or Node.js backend, React frontend, LMS integration with Moodle or Canvas, and video streaming. Database design with PostgreSQL.",
     "tags": ["edtech", "python", "nodejs", "react", "lms", "moodle", "postgresql"]},

    {"id": 89, "title": "R Developer", "company": "StatSoft", "location": "Remote",
     "description": "R developer needed for statistical computing and data analysis. Experience with R packages like ggplot2, dplyr, caret, and Shiny. Knowledge of Bayesian statistics and time series analysis preferred.",
     "tags": ["r", "statistics", "ggplot2", "shiny", "data analysis", "bayesian"]},

    {"id": 90, "title": "Platform Engineer", "company": "PlatformCo", "location": "Remote",
     "description": "Platform engineer needed to build internal developer platforms and tooling. Skills: Kubernetes, Terraform, CI/CD pipelines, Python, Go, and developer experience best practices. Experience with Backstage preferred.",
     "tags": ["platform engineering", "kubernetes", "terraform", "ci/cd", "golang", "developer experience"]},

    {"id": 91, "title": "E-commerce Manager", "company": "ShopBD", "location": "Dhaka",
     "description": "E-commerce manager needed to manage online store operations, product listings, and digital marketing. Experience with Shopify or WooCommerce, Google Merchant Center, and analytics tools required.",
     "tags": ["ecommerce", "shopify", "woocommerce", "digital marketing", "analytics", "product management"]},

    {"id": 92, "title": "NLP Data Annotator", "company": "LabelAI", "location": "Remote",
     "description": "NLP data annotator needed to label and annotate text datasets for machine learning models. Attention to detail, strong language skills in English and Bangla, and experience with annotation tools like Label Studio.",
     "tags": ["nlp", "data annotation", "labeling", "machine learning", "text", "label studio"]},

    {"id": 93, "title": "Solutions Architect", "company": "CloudSolve", "location": "Remote",
     "description": "Solutions architect needed to design end-to-end technical solutions for clients. Experience with AWS or Azure certifications, cloud-native design patterns, and ability to present to technical and non-technical stakeholders.",
     "tags": ["solutions architect", "aws", "azure", "cloud", "system design", "architecture"]},

    {"id": 94, "title": "Streaming Engineer", "company": "StreamFlow", "location": "Remote",
     "description": "Streaming engineer needed to build real-time data streaming pipelines. Experience with Apache Kafka, Flink, or Spark Streaming. Knowledge of cloud event streaming services and Python or Scala required.",
     "tags": ["kafka", "flink", "spark streaming", "real-time", "data engineering", "python", "scala"]},

    {"id": 95, "title": "Prompt Engineer", "company": "GenAI Studio", "location": "Remote",
     "description": "Prompt engineer needed to design and optimize prompts for large language models. Experience with ChatGPT, Claude, or similar LLMs. Understanding of few-shot learning, chain-of-thought, and evaluation metrics.",
     "tags": ["prompt engineering", "llm", "chatgpt", "ai", "nlp", "generative ai"]},

    {"id": 96, "title": "Logistics Software Developer", "company": "MoveFast", "location": "Dhaka",
     "description": "Software developer needed to build logistics and fleet management systems. Skills: Python or Java backend, REST APIs, geolocation APIs, Google Maps integration, and MySQL or PostgreSQL databases.",
     "tags": ["logistics", "python", "java", "rest api", "google maps", "geolocation", "mysql"]},

    {"id": 97, "title": "Figma Designer", "company": "PixelUI", "location": "Remote",
     "description": "Figma designer needed to create wireframes, prototypes, and high-fidelity UI designs. Strong visual design skills, understanding of design tokens, component libraries, and collaboration with developers.",
     "tags": ["figma", "ui design", "prototyping", "wireframe", "design system", "ux"]},

    {"id": 98, "title": "Cybersecurity Incident Responder", "company": "CyberGuard", "location": "Dhaka",
     "description": "Cybersecurity incident responder needed to investigate and contain security breaches. Skills: SIEM tools, digital forensics, malware analysis, network traffic analysis, and threat intelligence.",
     "tags": ["incident response", "cybersecurity", "siem", "forensics", "malware analysis", "threat intelligence"]},

    {"id": 99, "title": "API Integration Developer", "company": "ConnectAPI", "location": "Remote",
     "description": "API integration developer needed to connect third-party services and build middleware solutions. Experience with REST, GraphQL, SOAP, OAuth 2.0, and tools like MuleSoft or Boomi preferred.",
     "tags": ["api integration", "rest", "graphql", "soap", "mulesoft", "oauth", "middleware"]},

    {"id": 100, "title": "AI Product Manager", "company": "AIFirst", "location": "Dhaka",
     "description": "AI product manager needed to lead the development of AI-powered products. Understanding of machine learning concepts, ability to define AI product roadmaps, user research, and collaboration with data science teams.",
     "tags": ["product management", "ai", "machine learning", "roadmap", "user research", "agile"]},
]

print("Dataset loaded:", len(JOB_LISTINGS), "job listings.")

## Step 2 - Text Preprocessor

In [ ]:
# ============================================================
# CELL 3 - Text Preprocessing
# Pipeline: Tokenize -> Remove Stopwords -> Stem
# ============================================================

stemmer    = PorterStemmer()
STOP_WORDS = set(stopwords.words('english'))

def tokenize(text):
    """Lowercase, remove punctuation, split into tokens."""
    text = text.lower()
    text = re.sub(r'[^a-z\s]', ' ', text)
    return text.split()

def remove_stopwords(tokens):
    """Drop common English words like: the, is, and."""
    return [t for t in tokens if t not in STOP_WORDS]

def stem_tokens(tokens):
    """Reduce words to root form: running->run, developer->develop."""
    return [stemmer.stem(t) for t in tokens]

def preprocess(text):
    """Full preprocessing pipeline."""
    return stem_tokens(remove_stopwords(tokenize(text)))

# Quick demo
sample = "We are looking for an experienced Python developer"
print("Original :", sample)
print("Processed:", preprocess(sample))
print("\nPreprocessor ready.")

## Step 3 - Build the Inverted Index

In [ ]:
# ============================================================
# CELL 4 - Inverted Index
# Structure: term -> { doc_id: {frequency, positions} }
# ============================================================

class InvertedIndex:
    def __init__(self):
        self.index       = {}   # term -> {doc_id: {frequency, positions}}
        self.doc_count   = 0
        self.doc_lengths = {}   # doc_id -> token count

    def build(self, documents):
        self.doc_count = len(documents)
        for doc in documents:
            doc_id    = doc["id"]
            full_text = doc['title'] + ' ' + doc['description'] + ' ' + ' '.join(doc['tags'])
            tokens    = preprocess(full_text)
            self.doc_lengths[doc_id] = len(tokens)

            for pos, term in enumerate(tokens):
                if term not in self.index:
                    self.index[term] = {}
                if doc_id not in self.index[term]:
                    self.index[term][doc_id] = {"frequency": 0, "positions": []}
                self.index[term][doc_id]["frequency"] += 1
                self.index[term][doc_id]["positions"].append(pos)

        print("Index built:", self.doc_count, "documents |", len(self.index), "unique terms")

    def get_postings(self, term):
        return self.index.get(term, {})

    def document_frequency(self, term):
        return len(self.index.get(term, {}))

    def stats(self):
        avg = sum(self.doc_lengths.values()) / len(self.doc_lengths) if self.doc_lengths else 0
        return {
            "total_documents": self.doc_count,
            "unique_terms":    len(self.index),
            "avg_doc_length":  round(avg, 2)
        }


# Build the index
index = InvertedIndex()
index.build(JOB_LISTINGS)

# Show index stats
s = index.stats()
print("Total Documents :", s['total_documents'])
print("Unique Terms    :", s['unique_terms'])
print("Avg Doc Length  :", s['avg_doc_length'], "tokens")

# Peek at one entry
sample_term = preprocess("python")[0]
print("\nPostings for 'python':", index.get_postings(sample_term))

## Step 4 - Query Processor (Boolean + TF-IDF)

In [ ]:
# ============================================================
# CELL 5 - Query Processor
# - AND query: intersection of postings lists
# - OR  query: union of postings lists
# - Ranking  : TF-IDF score
# ============================================================

class QueryProcessor:
    def __init__(self, index, documents):
        self.index = index
        self.docs  = {doc["id"]: doc for doc in documents}

    # ---------- TF-IDF ----------
    def _tf(self, freq, doc_len):
        return freq / doc_len if doc_len else 0.0

    def _idf(self, term):
        df = self.index.document_frequency(term)
        if df == 0:
            return 0.0
        return math.log((self.index.doc_count + 1) / (df + 1)) + 1

    def _tfidf(self, term, doc_id):
        p = self.index.get_postings(term)
        if doc_id not in p:
            return 0.0
        return self._tf(p[doc_id]["frequency"],
                        self.index.doc_lengths.get(doc_id, 1)) * self._idf(term)

    def _score(self, doc_id, terms):
        return sum(self._tfidf(t, doc_id) for t in terms)

    # ---------- Boolean ----------
    def _and_query(self, terms):
        if not terms:
            return set()
        result = set(self.index.get_postings(terms[0]).keys())
        for t in terms[1:]:
            result &= set(self.index.get_postings(t).keys())
        return result

    def _or_query(self, terms):
        result = set()
        for t in terms:
            result |= set(self.index.get_postings(t).keys())
        return result

    # ---------- Snippet ----------
    def _snippet(self, description, query, length=160):
        words = query.lower().split()
        dl    = description.lower()
        best  = 0
        for w in words:
            p = dl.find(w)
            if p != -1:
                best = max(0, p - 30)
                break
        snip = description[best: best + length]
        if best > 0:
            snip = "..." + snip
        if best + length < len(description):
            snip += "..."
        return snip

    # ---------- Main search ----------
    def search(self, query, mode="OR", top_k=10):
        """
        Args:
            query  : raw query string
            mode   : 'AND' or 'OR'
            top_k  : max results to return
        Returns:
            list of result dicts sorted by TF-IDF score
        """
        terms = preprocess(query)
        if not terms:
            return []

        if mode.upper() == "AND":
            matching = self._and_query(terms)
        else:
            matching = self._or_query(terms)

        if not matching:
            return []

        scored = sorted(
            [(d, self._score(d, terms)) for d in matching],
            key=lambda x: x[1],
            reverse=True
        )

        results = []
        for doc_id, score in scored[:top_k]:
            doc = self.docs[doc_id]
            results.append({
                "rank":     len(results) + 1,
                "id":       doc_id,
                "title":    doc["title"],
                "company":  doc["company"],
                "location": doc["location"],
                "snippet":  self._snippet(doc["description"], query),
                "score":    round(score, 4),
                "tags":     doc["tags"]
            })
        return results


processor = QueryProcessor(index, JOB_LISTINGS)
print("Query processor ready.")

## Step 5 - Result Display Helper

In [ ]:
# ============================================================
# CELL 6 - Display search results
# ============================================================

def show_results(query, mode="OR", top_k=10):
    """Run a search and print results in a readable format."""
    results = processor.search(query, mode=mode, top_k=top_k)

    print("=" * 65)
    print("  Query :", repr(query), "  |  Mode :", mode, "  |  Top :", top_k)
    print("=" * 65)

    if not results:
        print("  No matching jobs found. Try OR mode or different keywords.")
        return

    print("  Found", len(results), "result(s):")
    print()
    for r in results:
        print("  [" + str(r['rank']) + "] " + r['title'])
        print("       Company  :", r['company'])
        print("       Location :", r['location'])
        print("       Score    :", r['score'])
        print("       Snippet  :", r['snippet'])
        print("       Tags     :", ", ".join(r['tags']))
        print()

print("Display helper ready.")

## Step 6 - Run Demo Searches

Change the queries below to test your own searches!

In [ ]:
# ============================================================
# CELL 7 - Demo: OR Query
# Returns jobs matching ANY of the query words
# ============================================================
show_results("python developer", mode="OR", top_k=5)

In [ ]:
# ============================================================
# CELL 8 - Demo: AND Query
# Returns jobs matching ALL query words
# ============================================================
show_results("machine learning python", mode="AND", top_k=5)

In [ ]:
# ============================================================
# CELL 9 - Demo: Single keyword
# ============================================================
show_results("android", mode="OR", top_k=5)

In [ ]:
# ============================================================
# CELL 10 - Demo: AND with specific terms
# ============================================================
show_results("cloud aws kubernetes", mode="AND", top_k=5)

In [ ]:
# ============================================================
# CELL 11 - Demo: Remote jobs related to data
# ============================================================
show_results("data remote", mode="OR", top_k=5)

## Step 7 - Interactive Search Widget

In [ ]:
# ============================================================
# CELL 12 - Interactive Search (ipywidgets)
# Use the text box and dropdown to search live!
# ============================================================
try:
    import ipywidgets as widgets
    from IPython.display import display, clear_output

    query_input = widgets.Text(
        value='python developer',
        placeholder='Type your search query...',
        description='Query:',
        layout=widgets.Layout(width='50%')
    )
    mode_input = widgets.Dropdown(
        options=['OR', 'AND'],
        value='OR',
        description='Mode:',
    )
    topk_input = widgets.IntSlider(
        value=5, min=1, max=20, step=1,
        description='Top K:',
    )
    btn    = widgets.Button(description='Search', button_style='primary')
    output = widgets.Output()

    def on_click(b):
        with output:
            clear_output()
            show_results(query_input.value, mode=mode_input.value, top_k=topk_input.value)

    btn.on_click(on_click)
    display(widgets.VBox([
        widgets.HBox([query_input, mode_input, topk_input]),
        btn,
        output
    ]))
except Exception as e:
    print("Widget not available (", e, "). Using show_results() directly instead.")
    show_results("python developer", mode="OR", top_k=5)

## Step 8 - Index Statistics

In [ ]:
# ============================================================
# CELL 13 - Index Statistics
# ============================================================
import collections

s = index.stats()
print("=" * 40)
print("  INDEX STATISTICS")
print("=" * 40)
print("  Total Documents :", s['total_documents'])
print("  Unique Terms    :", s['unique_terms'])
print("  Avg Doc Length  :", s['avg_doc_length'], "tokens")
print()

# Top 15 most frequent terms across all docs
term_totals = {
    term: sum(v["frequency"] for v in postings.values())
    for term, postings in index.index.items()
}
top15 = sorted(term_totals.items(), key=lambda x: x[1], reverse=True)[:15]

print("  Top 15 most common terms:")
print("  " + "-" * 35)
for term, freq in top15:
    bar = "#" * freq
    print("  " + term.ljust(15) + " " + bar + "  (" + str(freq) + ")")